# TrOCR Text Continuation Pipeline
**Context image → TrOCR (ViT encoder + mBART/RoBERTa decoder) → Predicted continuation text**

Notebook này tích hợp toàn bộ pipeline: `config` → `utils` → `dataset` → `model` → `metrics` → `train` → `inference`

---
## Mục lục
1. [Cài đặt & Import](#1)
2. [Configuration](#2)
3. [Utilities](#3)
4. [Dataset & DataLoader](#4)
5. [Model](#5)
6. [Metrics](#6)
7. [Training Loop](#7)
8. [Inference & Submission](#8)

---
## 1. Cài đặt & Import <a id='1'></a>

In [ ]:
# Cài đặt thư viện (bỏ comment nếu chạy trên Kaggle/Colab)
# !pip install transformers accelerate pillow pandas numpy torch -q

In [ ]:
import os
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw, ImageFont
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

---
## 2. Configuration <a id='2'></a>

Toàn bộ hyperparameter và đường dẫn file được quản lý tập trung tại đây.
Điều chỉnh `get_config()` để thay đổi thực nghiệm.

In [ ]:
# ============================================================
# config.py
# ============================================================

LANGUAGES = ["English", "French", "German", "Italian", "Spanish"]


def get_config() -> dict:
    root = Path(".")
    return {
        "seed": 42,
        "train_file": str(root / "train-3.csv"),
        "test_file": str(root / "test.csv"),
        "output_dir": str(root / "outputs"),
        "debug_dir": str(root / "debug_outputs"),
        # Model
        "model_name": "microsoft/trocr-base-printed",
        # Rendering
        "font_size": 18,
        "padding": 12,
        "image_size": 384,
        "context_tail_chars": 160,
        # Encoder / Decoder tuning
        "freeze_encoder": True,
        "decoder_tune_mode": "top_layers",
        "decoder_train_layers": 4,
        # Training
        "val_split": 0.10,
        "epochs": 10,
        "batch_size": 4,
        "grad_accum_steps": 2,
        "lr": 1.0e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.10,
        "grad_clip": 1.0,
        "bf16": True,
        # Generation
        "max_target_length": 128,
        "max_new_tokens": 72,
        "min_new_tokens": 8,
        "num_beams": 2,
        "repetition_penalty": 1.20,
        "no_repeat_ngram_size": 3,
        "length_penalty": 0.8,
        # Loss weighting
        "prefix_token_count": 12,
        "prefix_loss_weight": 3.0,
        "prompt_loss_weight": 1.0,
        # Debug / Eval
        "debug_topk": 5,
        "eval_samples": 24,
        "debug_image_count": 4,
        "log_interval": 20,
        "early_stop_patience": 4,
        "early_stop_warmup_epochs": 2,
    }


def resolve_config(cfg: dict) -> dict:
    """Clamp / cast mọi giá trị để đảm bảo hợp lệ."""
    cfg = dict(cfg)
    cfg["epochs"] = int(max(1, cfg.get("epochs", 10)))
    cfg["batch_size"] = int(max(1, cfg.get("batch_size", 4)))
    cfg["grad_accum_steps"] = int(max(1, cfg.get("grad_accum_steps", 2)))
    cfg["image_size"] = int(max(128, cfg.get("image_size", 384)))
    cfg["context_tail_chars"] = int(max(32, cfg.get("context_tail_chars", 160)))
    cfg["decoder_train_layers"] = int(max(1, cfg.get("decoder_train_layers", 4)))
    cfg["max_target_length"] = int(max(16, cfg.get("max_target_length", 128)))
    cfg["max_new_tokens"] = int(max(8, cfg.get("max_new_tokens", 72)))
    cfg["min_new_tokens"] = int(max(0, min(cfg["max_new_tokens"] - 1, cfg.get("min_new_tokens", 8))))
    cfg["num_beams"] = int(max(1, cfg.get("num_beams", 4)))
    cfg["no_repeat_ngram_size"] = int(max(0, cfg.get("no_repeat_ngram_size", 3)))
    cfg["prefix_token_count"] = int(max(1, cfg.get("prefix_token_count", 12)))
    cfg["prompt_loss_weight"] = float(max(0.0, cfg.get("prompt_loss_weight", 1.0)))
    cfg["debug_topk"] = int(max(1, cfg.get("debug_topk", 5)))
    cfg["eval_samples"] = int(max(1, cfg.get("eval_samples", 24)))
    cfg["debug_image_count"] = int(max(1, cfg.get("debug_image_count", 4)))
    cfg["log_interval"] = int(max(1, cfg.get("log_interval", 20)))
    cfg["early_stop_patience"] = int(max(1, cfg.get("early_stop_patience", 4)))
    cfg["early_stop_warmup_epochs"] = int(max(1, cfg.get("early_stop_warmup_epochs", 2)))
    return cfg


# Khởi tạo config toàn cục
config = resolve_config(get_config())
print("Config loaded:")
for k, v in config.items():
    print(f"  {k:30s} = {v}")

---
## 3. Utilities <a id='3'></a>

Các hàm tiện ích: render text → PIL image, fit image vào canvas vuông, token/pixel helpers.

In [ ]:
# ============================================================
# utils.py
# ============================================================

LANG_PROMPTS = {
    "English": "English: ",
    "French": "French: ",
    "German": "German: ",
    "Italian": "Italian: ",
    "Spanish": "Spanish: ",
}


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str | os.PathLike) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def get_font(font_size: int = 18):
    font_paths = [
        "C:/Windows/Fonts/times.ttf",
        "C:/Windows/Fonts/arial.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
    ]
    for path in font_paths:
        if os.path.exists(path):
            return ImageFont.truetype(path, font_size)
    return ImageFont.load_default()


def render_text_to_image(text: str, max_width: int, font_size: int = 18, padding: int = 12) -> Image.Image:
    """Render text thành ảnh grayscale với word-wrap theo max_width."""
    text = str(text or "")
    font = get_font(font_size)
    content_width = max(16, int(max_width) - 2 * padding)

    lines = []
    current = ""
    for word in text.split(" "):
        probe = f"{current} {word}".strip()
        bbox = font.getbbox(probe)
        if bbox[2] - bbox[0] <= content_width:
            current = probe
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    if not lines:
        lines = ["."]

    line_h = font_size + 4
    img = Image.new("L", (int(max_width), len(lines) * line_h + 2 * padding), 0)
    draw = ImageDraw.Draw(img)
    y = padding
    for line in lines:
        draw.text((padding, y), line, fill=255, font=font)
        y += line_h
    return img


def fit_image_to_canvas(img: Image.Image, image_size: int, anchor: str = "bottom") -> Image.Image:
    """Fit ảnh vào canvas vuông image_size×image_size (crop nếu quá lớn)."""
    if img.mode != "RGB":
        img = img.convert("RGB")
    canvas = Image.new("RGB", (image_size, image_size), (0, 0, 0))
    src = img
    if src.size[0] > image_size:
        src = src.crop((max(0, src.size[0] - image_size), 0, src.size[0], src.size[1]))
    if src.size[1] > image_size:
        if anchor == "bottom":
            src = src.crop((0, src.size[1] - image_size, src.size[0], src.size[1]))
        else:
            src = src.crop((0, 0, src.size[0], image_size))
    x = 0
    y = image_size - src.size[1] if anchor == "bottom" else 0
    canvas.paste(src, (x, y))
    return canvas


def white_pixel_density(img: Image.Image, threshold: int = 127) -> float:
    arr = np.array(img.convert("L"))
    return float((arr > threshold).mean())


def image_to_pixels(img: Image.Image, sample_id: int, threshold: int = 127) -> np.ndarray:
    arr = np.array(img)
    rows, cols = np.where(arr > threshold)
    if len(rows) == 0:
        return np.array([[sample_id, 0, 0]], dtype=np.int64)
    return np.stack(
        [
            np.full(len(rows), sample_id, dtype=np.int64),
            rows.astype(np.int64),
            cols.astype(np.int64),
        ],
        axis=1,
    )


def clean_text(text: str) -> str:
    return " ".join((text or "").split()).strip()


def get_language_prompt(language: str) -> str:
    return LANG_PROMPTS.get(str(language), f"{language}: ")


def tail_context(text: str, max_chars: int) -> str:
    """Lấy max_chars ký tự cuối của context (giảm độ dài ảnh render)."""
    text = str(text or "")
    return text if len(text) <= max_chars else text[-max_chars:]


def clean_prediction(text: str, language: str) -> str:
    """Loại bỏ prefix prompt khỏi output generated."""
    text = clean_text(text)
    lower = text.lower()
    for prompt in LANG_PROMPTS.values():
        prefix = prompt.lower()
        if lower.startswith(prefix):
            text = clean_text(text[len(prompt):])
            lower = text.lower()
    for prefix in ["answer:", "response:", "output:", "assistant:"]:
        if lower.startswith(prefix):
            text = clean_text(text[len(prefix):])
            lower = text.lower()
    lang_prompt = get_language_prompt(language)
    if text.startswith(lang_prompt):
        text = clean_text(text[len(lang_prompt):])
    return text


# ── Quick sanity test ──────────────────────────────────────────
set_seed(config["seed"])
sample_img = render_text_to_image("Hello World! This is a test.", max_width=300)
canvas_img = fit_image_to_canvas(sample_img, config["image_size"])
print(f"render_text_to_image → {sample_img.size}")
print(f"fit_image_to_canvas  → {canvas_img.size}")
canvas_img  # hiển thị ảnh trong notebook

---
## 4. Dataset & DataLoader <a id='4'></a>

- `stratified_split`: chia train/val theo phân phối ngôn ngữ
- `OCRDataset`: render context → ảnh → pixel_values + tokenize target
- `collate_fn`: pad sequences trong batch

In [ ]:
# ============================================================
# dataset.py
# ============================================================

def stratified_split(df: pd.DataFrame, col: str, frac: float, seed: int = 42):
    """Chia stratified theo cột col với tỷ lệ val = frac."""
    val_parts = []
    for _, g in df.groupby(col, group_keys=False):
        n = max(1, int(round(len(g) * frac))) if len(g) > 1 else 1
        n = min(n, len(g) - 1) if len(g) > 1 else 1
        val_parts.append(g.sample(n=n, random_state=seed))
    raw = pd.concat(val_parts)
    val = raw.sample(frac=1, random_state=seed).reset_index(drop=True)
    train = df.drop(raw.index).sample(frac=1, random_state=seed).reset_index(drop=True)
    return train, val


class OCRDataset(Dataset):
    def __init__(self, df: pd.DataFrame, processor, config: dict, is_test: bool = False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.cfg = config
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        full_context = str(row["context"])
        context = tail_context(full_context, int(self.cfg["context_tail_chars"]))
        language = str(row["language"])
        max_width = int(row["max_width"])

        # Render context text → ảnh → tensor
        raw_img = render_text_to_image(context, int(self.cfg["image_size"]), self.cfg["font_size"], self.cfg["padding"])
        img = fit_image_to_canvas(raw_img, self.cfg["image_size"], anchor="bottom")
        pixel_values = self.processor(images=img, return_tensors="pt").pixel_values.squeeze(0)

        sample = {
            "pixel_values": pixel_values,
            "context_text": context,
            "full_context_text": full_context,
            "context_char_len": len(context),
            "full_context_char_len": len(full_context),
            "language": language,
            "max_width": max_width,
            "orig_render_height": int(raw_img.size[1]),
            "orig_render_width": int(raw_img.size[0]),
            "render_height": int(img.size[1]),
            "render_width": int(img.size[0]),
            "resize_ratio_w": float(img.size[0] / max(raw_img.size[0], 1)),
            "resize_ratio_h": float(min(1.0, img.size[1] / max(raw_img.size[1], 1))),
            "context_white_density": white_pixel_density(raw_img),
            "pad_token_id": int(self.processor.tokenizer.pad_token_id),
        }

        if not self.is_test:
            target = str(row["target"])
            tokenizer = self.processor.tokenizer
            prompt_text = get_language_prompt(language)
            prompt_ids = tokenizer(
                prompt_text, add_special_tokens=False, return_tensors="pt"
            ).input_ids.squeeze(0)
            target_ids = tokenizer(
                target,
                add_special_tokens=False,
                truncation=True,
                max_length=max(1, self.cfg["max_target_length"] - len(prompt_ids) - 1),
                return_tensors="pt",
            ).input_ids.squeeze(0)
            start_id = int(tokenizer.cls_token_id)
            eos_id = int(tokenizer.sep_token_id)
            sample["decoder_input_ids"] = torch.cat([
                torch.tensor([start_id], dtype=torch.long),
                prompt_ids.long(),
                target_ids.long(),
            ])
            sample["labels"] = torch.cat([
                prompt_ids.long(),
                target_ids.long(),
                torch.tensor([eos_id], dtype=torch.long),
            ])
            sample["prompt_text"] = prompt_text
            sample["prompt_ids"] = prompt_ids.long()
            sample["prompt_token_len"] = int(len(prompt_ids))
            sample["target_text"] = target
            sample["target_char_len"] = len(target)
        else:
            tokenizer = self.processor.tokenizer
            prompt_text = get_language_prompt(language)
            prompt_ids = tokenizer(
                prompt_text, add_special_tokens=False, return_tensors="pt"
            ).input_ids.squeeze(0)
            sample["prompt_text"] = prompt_text
            sample["prompt_ids"] = prompt_ids.long()

        if "id" in row.index:
            sample["id"] = int(row["id"])
        return sample


def collate_fn(batch: list[dict]) -> dict:
    out = {}
    tensor_keys = {k for k, v in batch[0].items() if isinstance(v, torch.Tensor)}
    meta_keys = set(batch[0].keys()) - tensor_keys
    for key in tensor_keys:
        values = [b[key] for b in batch]
        if key == "labels":
            out[key] = pad_sequence(values, batch_first=True, padding_value=-100)
        elif key in {"decoder_input_ids", "prompt_ids"}:
            out[key] = pad_sequence(values, batch_first=True, padding_value=int(batch[0]["pad_token_id"]))
        else:
            out[key] = torch.stack(values)
    for key in meta_keys:
        out[key] = [b[key] for b in batch]
    return out


print("Dataset classes defined. Load data in cell bên dưới sau khi có file CSV.")

In [ ]:
# ── Load dữ liệu (thay đường dẫn nếu cần) ─────────────────────
train_df = pd.read_csv(config["train_file"])
print(f"Total samples : {len(train_df)}")
print(f"Language dist : {train_df['language'].value_counts().to_dict()}")
print(train_df.head(3))

---
## 5. Model <a id='5'></a>

- `build_processor`: TrOCRProcessor (feature extractor + tokenizer)
- `build_model`: VisionEncoderDecoderModel với encoder frozen, decoder tuned top-k layers
- `save_bundle` / `load_bundle`: lưu & load checkpoint

In [ ]:
# ============================================================
# model.py
# ============================================================

def build_processor(model_name: str):
    return TrOCRProcessor.from_pretrained(model_name)


def freeze_decoder_top_layers(model, num_layers: int) -> dict:
    """Freeze toàn bộ decoder, sau đó unfreeze num_layers layer cuối + embeddings."""
    decoder = model.decoder.model.decoder
    layers = decoder.layers
    keep_from = max(0, len(layers) - int(num_layers))

    total_decoder = 0
    trainable_decoder = 0
    unfrozen_names = []

    # Freeze tất cả
    for name, param in model.decoder.named_parameters():
        total_decoder += param.numel()
        param.requires_grad = False

    # Unfreeze top layers
    for idx, layer in enumerate(layers):
        if idx < keep_from:
            continue
        for name, param in layer.named_parameters():
            param.requires_grad = True
            trainable_decoder += param.numel()
            unfrozen_names.append(f"layers.{idx}.{name}")

    # Unfreeze embeddings & LM head
    extra_modules = []
    dec = model.decoder.model.decoder
    if hasattr(dec, "layernorm_embedding"):
        extra_modules.append(("decoder.layernorm_embedding", dec.layernorm_embedding))
    if hasattr(dec, "embed_positions"):
        extra_modules.append(("decoder.embed_positions", dec.embed_positions))
    if hasattr(dec, "embed_tokens"):
        extra_modules.append(("decoder.embed_tokens", dec.embed_tokens))
    if hasattr(model.decoder, "output_projection"):
        extra_modules.append(("output_projection", model.decoder.output_projection))
    if hasattr(model.decoder, "lm_head"):
        extra_modules.append(("lm_head", model.decoder.lm_head))

    for prefix, module in extra_modules:
        for name, param in module.named_parameters():
            if not param.requires_grad:
                param.requires_grad = True
                trainable_decoder += param.numel()
                unfrozen_names.append(f"{prefix}.{name}")

    return {
        "mode": "top_layers",
        "decoder_total": total_decoder,
        "decoder_trainable": trainable_decoder,
        "decoder_kept_layers": list(range(keep_from, len(layers))),
        "decoder_unfrozen_count": len(unfrozen_names),
        "decoder_unfrozen_preview": unfrozen_names[:16],
    }


def build_model(cfg: dict, processor, device: torch.device):
    dtype = torch.bfloat16 if cfg.get("bf16") and device.type == "cuda" else torch.float32
    model = VisionEncoderDecoderModel.from_pretrained(cfg["model_name"], torch_dtype=dtype)
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.eos_token_id = processor.tokenizer.sep_token_id
    model.config.vocab_size = model.config.decoder.vocab_size
    model.generation_config.decoder_start_token_id = model.config.decoder_start_token_id
    model.generation_config.pad_token_id = model.config.pad_token_id
    model.generation_config.eos_token_id = model.config.eos_token_id
    if cfg.get("freeze_encoder", True):
        for param in model.encoder.parameters():
            param.requires_grad = False
    freeze_info = None
    if cfg.get("decoder_tune_mode") == "top_layers":
        freeze_info = freeze_decoder_top_layers(model, int(cfg.get("decoder_train_layers", 4)))
    model._freeze_info = freeze_info
    return model.to(device)


def save_bundle(model, processor, cfg: dict, save_dir: str) -> None:
    model.save_pretrained(save_dir)
    processor.save_pretrained(save_dir)
    with open(os.path.join(save_dir, "run_config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)


def load_bundle(save_dir: str, device: torch.device):
    processor = TrOCRProcessor.from_pretrained(save_dir)
    model = VisionEncoderDecoderModel.from_pretrained(save_dir)
    return model.to(device).eval(), processor


print("Model functions defined.")

In [ ]:
# ── Khởi tạo processor & model ────────────────────────────────
print("[1/2] Loading processor...")
processor = build_processor(config["model_name"])

print("[2/2] Building model...")
model = build_model(config, processor, device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100.0 * trainable / max(total, 1):.2f}%)")

if getattr(model, "_freeze_info", None):
    info = model._freeze_info
    print(f"Decoder tune mode : {info['mode']}")
    print(f"Kept layers       : {info['decoder_kept_layers']}")
    print(f"Trainable decoder : {info['decoder_trainable']:,} / {info['decoder_total']:,}")
    print(f"Unfrozen preview  : {info['decoder_unfrozen_preview'][:8]}")

---
## 6. Metrics <a id='6'></a>

Evaluation metrics:
- **CER** (Character Error Rate) — metric chính
- **Pixel F1** — render pred & true → so sánh pixel trắng
- **Prefix CER** — CER trên 16 ký tự đầu
- **Repetition ratio**, **Length ratio**, **Language consistency**

In [ ]:
# ============================================================
# metrics.py
# ============================================================

LANGUAGE_MARKERS = {
    "English": [" and ", " the ", " of ", " to ", " is ", " for ", " with "],
    "French":  [" le ", " la ", " les ", " de ", " des ", " une ", " est ", " pour "],
    "German":  [" der ", " die ", " und ", " das ", " ist ", " mit ", " eine "],
    "Italian": [" il ", " la ", " che ", " per ", " una ", " dei ", " gli ", " sono "],
    "Spanish": [" el ", " la ", " de ", " que ", " los ", " una ", " para ", " con "],
}
ENGLISH_FUNCTION_OPENERS = {"and", "the", "to", "for", "of", "in", "is"}


def char_error_rate(pred: str, target: str) -> float:
    """CER bằng dynamic programming O(|pred|*|target|)."""
    a, b = clean_text(pred), clean_text(target)
    if not b:
        return 0.0 if not a else 1.0
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        prev = dp[0]
        dp[0] = i
        for j, cb in enumerate(b, start=1):
            old = dp[j]
            cost = 0 if ca == cb else 1
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
            prev = old
    return dp[-1] / max(len(b), 1)


def repetition_ratio(text: str, n: int = 3) -> float:
    tokens = clean_text(text).split()
    if len(tokens) < n:
        return 0.0
    grams = [" ".join(tokens[i: i + n]) for i in range(len(tokens) - n + 1)]
    return 1.0 - (len(set(grams)) / max(len(grams), 1))


def length_ratio(pred: str, target: str) -> float:
    return len(clean_text(pred)) / max(len(clean_text(target)), 1)


def pixel_f1(pred: str, target: str, max_width: int, font_size: int, padding: int) -> float:
    pred_img = render_text_to_image(pred, max_width, font_size, padding)
    true_img = render_text_to_image(target, max_width, font_size, padding)
    canvas_h = max(pred_img.size[1], true_img.size[1], max_width)
    pred_img = fit_image_to_canvas(pred_img, canvas_h, anchor="top").convert("L")
    true_img = fit_image_to_canvas(true_img, canvas_h, anchor="top").convert("L")
    p = np.array(pred_img) > 127
    t = np.array(true_img) > 127
    tp = float((p & t).sum())
    fp = float((p & ~t).sum())
    fn = float((~p & t).sum())
    return (2.0 * tp + 1e-6) / (2.0 * tp + fp + fn + 1e-6)


def prefix_cer(pred: str, target: str, n: int = 16) -> float:
    return char_error_rate(clean_text(pred)[:n], clean_text(target)[:n])


def first_token(text: str) -> str:
    tokens = clean_text(text).split()
    return tokens[0].lower() if tokens else ""


def language_marker_counts(text: str) -> dict:
    cleaned = f" {clean_text(text).lower()} "
    return {lang: sum(cleaned.count(m) for m in markers)
            for lang, markers in LANGUAGE_MARKERS.items()}


def language_consistency_score(text: str, expected_language: str) -> float:
    counts = language_marker_counts(text)
    total = sum(counts.values())
    if total == 0 or expected_language not in counts:
        return 0.0
    return float(counts[expected_language] / total)


def english_opener_flag(text: str) -> float:
    return float(first_token(text) in ENGLISH_FUNCTION_OPENERS)


def summarize_prediction(pred: str, target: str, max_width: int, font_size: int,
                          padding: int, expected_language: str) -> dict:
    pred_clean, target_clean = clean_text(pred), clean_text(target)
    lang_match = pred_clean[:1].isalpha() and target_clean[:1].isalpha()
    marker_counts = language_marker_counts(pred_clean)
    best_marker_language = max(marker_counts, key=marker_counts.get) if marker_counts else expected_language
    fst = first_token(pred_clean)
    return {
        "cer": char_error_rate(pred_clean, target_clean),
        "prefix_cer": prefix_cer(pred_clean, target_clean),
        "pixel_f1": pixel_f1(pred_clean, target_clean, max_width, font_size, padding),
        "repeat": repetition_ratio(pred_clean),
        "len_ratio": length_ratio(pred_clean, target_clean),
        "pred_char_len": len(pred_clean),
        "target_char_len": len(target_clean),
        "alpha_start_match": float(lang_match),
        "first_token": fst,
        "lang_consistency": language_consistency_score(pred_clean, expected_language),
        "best_marker_language": best_marker_language,
        "marker_english": marker_counts["English"],
        "marker_french": marker_counts["French"],
        "marker_german": marker_counts["German"],
        "marker_italian": marker_counts["Italian"],
        "marker_spanish": marker_counts["Spanish"],
    }


# ── Quick test ─────────────────────────────────────────────────
print("CER test:", char_error_rate("hello world", "hello world"))  # 0.0
print("CER test:", char_error_rate("helo world", "hello world"))   # nhỏ
print("LangConsistency:", language_consistency_score("the quick brown fox and the lazy dog", "English"))

---
## 7. Training Loop <a id='7'></a>

Flow:
1. `build_optimizer` → AdamW + LinearWarmup + CosineAnnealing
2. `run_epoch` → forward pass với weighted cross-entropy
3. `evaluate_generation` → generate + tính metrics trên val set
4. Early stopping theo CER

In [ ]:
# ============================================================
# train.py — helpers
# ============================================================

def build_optimizer(model, cfg: dict, total_steps: int):
    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
    )
    warmup = max(1, int(total_steps * cfg["warmup_ratio"]))
    decay = max(1, total_steps - warmup)
    scheduler = SequentialLR(
        optimizer,
        schedulers=[
            LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup),
            CosineAnnealingLR(optimizer, T_max=decay, eta_min=1e-7),
        ],
        milestones=[warmup],
    )
    return optimizer, scheduler


def describe_batch(batch: dict) -> dict:
    pixel_values = batch["pixel_values"]
    labels = batch["labels"]
    valid_label_counts = (labels != -100).sum(dim=1).float()
    stats = {
        "pixel_shape": tuple(pixel_values.shape),
        "pixel_min": float(pixel_values.min()),
        "pixel_max": float(pixel_values.max()),
        "pixel_mean": float(pixel_values.mean()),
        "pixel_std": float(pixel_values.std()),
        "label_tokens_mean": float(valid_label_counts.mean()),
        "label_tokens_min": int(valid_label_counts.min()),
        "label_tokens_max": int(valid_label_counts.max()),
    }
    if "target_char_len" in batch:
        tgt = np.array(batch["target_char_len"], dtype=np.float32)
        ctx = np.array(batch["context_char_len"], dtype=np.float32)
        stats.update({
            "target_chars_mean": float(tgt.mean()),
            "context_chars_mean": float(ctx.mean()),
            "orig_render_h_mean": float(np.mean(batch["orig_render_height"])),
            "orig_render_w_mean": float(np.mean(batch["orig_render_width"])),
            "render_h_mean": float(np.mean(batch["render_height"])),
            "max_width_mean": float(np.mean(batch["max_width"])),
            "resize_ratio_h_mean": float(np.mean(batch["resize_ratio_h"])),
            "resize_ratio_w_mean": float(np.mean(batch["resize_ratio_w"])),
            "white_density_mean": float(np.mean(batch["context_white_density"])),
        })
    return stats


def compute_weighted_loss(logits, labels, prompt_lengths,
                           prefix_token_count, prefix_loss_weight, prompt_loss_weight):
    """
    Weighted cross-entropy:
    - Prompt tokens  : trọng số prompt_loss_weight
    - Prefix target  : trọng số prefix_loss_weight (khuyến khích đúng đầu câu)
    - Remaining      : trọng số 1.0
    """
    vocab = logits.size(-1)
    flat_loss = F.cross_entropy(
        logits.view(-1, vocab), labels.view(-1), ignore_index=-100, reduction="none"
    ).view_as(labels)

    valid = labels != -100
    weights = torch.ones_like(flat_loss)
    prompt_mask = torch.zeros_like(valid)
    for i in range(labels.size(0)):
        prompt_len = int(prompt_lengths[i])
        if prompt_len > 0:
            prompt_mask[i, :min(prompt_len, labels.size(1))] = True

    prompt_mask = prompt_mask & valid
    target_valid = valid & (~prompt_mask)

    if prompt_loss_weight != 1.0:
        weights = torch.where(prompt_mask, torch.full_like(weights, float(prompt_loss_weight)), weights)

    prefix_mask = torch.zeros_like(valid)
    if prefix_loss_weight > 1.0 and prefix_token_count > 0:
        for i in range(labels.size(0)):
            valid_idx = torch.nonzero(target_valid[i], as_tuple=False).squeeze(-1)
            if valid_idx.numel() == 0:
                continue
            take = valid_idx[:prefix_token_count]
            prefix_mask[i, take] = True
        weights = torch.where(prefix_mask, torch.full_like(weights, float(prefix_loss_weight)), weights)

    weighted = flat_loss * weights * valid
    denom = (weights * valid).sum().clamp_min(1.0)
    loss = weighted.sum() / denom

    nonprefix_mask = target_valid & (~prefix_mask)
    prompt_loss = (flat_loss * prompt_mask).sum() / prompt_mask.sum().clamp_min(1)
    prefix_loss_ = (flat_loss * prefix_mask).sum() / prefix_mask.sum().clamp_min(1)
    nonprefix_loss = (flat_loss * nonprefix_mask).sum() / nonprefix_mask.sum().clamp_min(1)
    return loss, float(prompt_loss), float(prefix_loss_), float(nonprefix_loss), int(prefix_mask.sum())


@torch.no_grad()
def debug_first_step(model, processor, sample: dict, device, topk: int = 5):
    """Hiển thị top-k token tiếp theo tại bước đầu tiên (debug)."""
    pixel_values = sample["pixel_values"].unsqueeze(0).to(device)
    start_id = int(processor.tokenizer.cls_token_id)
    decoder_prompt_ids = torch.cat([
        torch.tensor([start_id], device=device, dtype=torch.long),
        sample["prompt_ids"].to(device),
    ]).unsqueeze(0)
    outputs = model(pixel_values=pixel_values, decoder_input_ids=decoder_prompt_ids)
    logits = outputs.logits[:, -1, :]
    probs = torch.softmax(logits, dim=-1)
    vals, ids = torch.topk(probs, k=min(int(topk), probs.size(-1)), dim=-1)
    parts = []
    for prob, tok_id in zip(vals[0].tolist(), ids[0].tolist()):
        tok = processor.tokenizer.decode([int(tok_id)]).replace("\n", "\\n").strip() or "<blank>"
        parts.append(f"{tok}:{prob:.2f}")
    return " | ".join(parts)


def run_epoch(model, loader, optimizer, scheduler, cfg, device, tag, train_mode) -> float:
    model.train() if train_mode else model.eval()
    use_bf16 = bool(cfg.get("bf16") and device.type == "cuda")
    losses = []
    grad_accum = int(cfg["grad_accum_steps"])
    if train_mode:
        optimizer.zero_grad(set_to_none=True)

    for bi, batch in enumerate(loader):
        if bi == 0:
            stats = describe_batch(batch)
            print(f"  [{tag}] batch shape={stats['pixel_shape']} "
                  f"pix=({stats['pixel_min']:.3f},{stats['pixel_max']:.3f}) "
                  f"labelTok={stats['label_tokens_mean']:.1f}")

        pixel_values = batch["pixel_values"].to(device)
        decoder_input_ids = batch["decoder_input_ids"].to(device)
        labels = batch["labels"].to(device)

        with torch.set_grad_enabled(train_mode):
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
                outputs = model(
                    pixel_values=pixel_values,
                    decoder_input_ids=decoder_input_ids,
                    labels=labels,
                )
                loss, prompt_loss, prefix_loss, nonprefix_loss, prefix_count = compute_weighted_loss(
                    outputs.logits, labels, batch["prompt_token_len"],
                    int(cfg["prefix_token_count"]), float(cfg["prefix_loss_weight"]), float(cfg["prompt_loss_weight"]),
                )

        if not torch.isfinite(loss):
            print(f"  [{tag}] non-finite loss at batch {bi}, skip")
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
            continue

        if train_mode:
            (loss / grad_accum).backward()

        losses.append(float(loss.item()))

        if train_mode:
            do_step = (bi + 1) % grad_accum == 0 or (bi + 1) == len(loader)
            if do_step:
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if scheduler:
                    scheduler.step()
            if bi % int(cfg["log_interval"]) == 0:
                lr = scheduler.get_last_lr()[-1] if scheduler else cfg["lr"]
                print(f"  [{tag}] B{bi:>3d}/{len(loader)} loss={loss.item():.4f} "
                      f"prompt={prompt_loss:.4f} prefix={prefix_loss:.4f} tail={nonprefix_loss:.4f} lr={lr:.2e}")

    return float(np.mean(losses)) if losses else float("nan")


@torch.no_grad()
def evaluate_generation(model, processor, dataset, cfg, device, epoch) -> dict:
    model.eval()
    limit = min(int(cfg["eval_samples"]), len(dataset))
    debug_dir = ensure_dir(os.path.join(cfg["debug_dir"], "train", f"epoch_{epoch:03d}"))
    rows = []

    for i in range(limit):
        sample = dataset[i]
        pixel_values = sample["pixel_values"].unsqueeze(0).to(device)
        start_id = int(processor.tokenizer.cls_token_id)
        decoder_prompt_ids = torch.cat([
            torch.tensor([start_id], device=device, dtype=torch.long),
            sample["prompt_ids"].to(device),
        ]).unsqueeze(0)

        generated_ids = model.generate(
            pixel_values=pixel_values,
            decoder_input_ids=decoder_prompt_ids,
            max_new_tokens=cfg["max_new_tokens"],
            min_new_tokens=cfg["min_new_tokens"],
            num_beams=cfg["num_beams"],
            early_stopping=True,
            repetition_penalty=float(cfg["repetition_penalty"]),
            no_repeat_ngram_size=int(cfg["no_repeat_ngram_size"]),
            length_penalty=float(cfg["length_penalty"]),
        )
        pred = clean_prediction(
            processor.batch_decode(generated_ids, skip_special_tokens=True)[0],
            sample["language"],
        )
        true = sample["target_text"]
        summary = summarize_prediction(
            pred, true, int(sample["max_width"]),
            cfg["font_size"], cfg["padding"], sample["language"],
        )
        rows.append({"language": sample["language"], "pred": pred, "true": true,
                     "max_width": int(sample["max_width"]), **summary})

    df = pd.DataFrame(rows)
    per_lang = df.groupby("language")["cer"].mean().to_dict() if not df.empty else {}
    per_lang_pix = df.groupby("language")["pixel_f1"].mean().to_dict() if not df.empty else {}
    unique_pred_ratio = df["pred"].nunique() / max(len(df), 1) if not df.empty else 0.0
    alpha_start_match = float(df["alpha_start_match"].mean()) if not df.empty else 0.0
    lang_consistency = float(df["lang_consistency"].mean()) if not df.empty else 0.0
    first_token_mode = df["first_token"].mode().iat[0] if not df.empty and not df["first_token"].mode().empty else ""
    first_token_top = df["first_token"].value_counts().head(5).to_dict() if not df.empty else {}

    first_token_by_lang, english_opener_rate = {}, {}
    if not df.empty:
        eng_openers = {"and", "the", "to", "for", "of", "in", "is"}
        for lang, grp in df.groupby("language"):
            mode = grp["first_token"].mode()
            first_token_by_lang[lang] = mode.iat[0] if not mode.empty else ""
            english_opener_rate[lang] = float(grp["first_token"].isin(eng_openers).mean())

    dbg_first = debug_first_step(model, processor, dataset[0], device, topk=int(cfg["debug_topk"]))

    # Lưu debug images
    for i, row in enumerate(rows[:int(cfg["debug_image_count"])]):
        render_text_to_image(dataset[i]["context_text"], row["max_width"], cfg["font_size"], cfg["padding"]).save(
            debug_dir / f"context_{i}.png")
        render_text_to_image(row["true"], row["max_width"], cfg["font_size"], cfg["padding"]).save(
            debug_dir / f"target_{i}.png")
        render_text_to_image(row["pred"], row["max_width"], cfg["font_size"], cfg["padding"]).save(
            debug_dir / f"pred_{i}.png")

    return {
        "cer_mean": float(df["cer"].mean()) if not df.empty else float("inf"),
        "pixel_f1": float(df["pixel_f1"].mean()) if not df.empty else 0.0,
        "prefix_cer": float(df["prefix_cer"].mean()) if not df.empty else float("inf"),
        "repeat": float(df["repeat"].mean()) if not df.empty else 0.0,
        "len_ratio": float(df["len_ratio"].mean()) if not df.empty else 0.0,
        "pred_len": float(df["pred"].str.len().mean()) if not df.empty else 0.0,
        "true_len": float(df["true"].str.len().mean()) if not df.empty else 0.0,
        "short_pct": 100.0 * float((df["len_ratio"] < 0.5).mean()) if not df.empty else 0.0,
        "long_pct": 100.0 * float((df["len_ratio"] > 1.5).mean()) if not df.empty else 0.0,
        "unique_pred_ratio": float(unique_pred_ratio),
        "alpha_start_match": alpha_start_match,
        "lang_consistency": lang_consistency,
        "first_token_mode": first_token_mode,
        "first_token_top": first_token_top,
        "first_token_by_lang": first_token_by_lang,
        "english_opener_rate": english_opener_rate,
        "cer_per_lang": per_lang,
        "pix_per_lang": per_lang_pix,
        "examples": rows[:4],
        "debug_dir": str(debug_dir),
        "debug_first_step": dbg_first,
    }


print("Training functions defined.")

In [ ]:
# ============================================================
# train.py — main train() function
# ============================================================

def train(cfg: dict = None):
    if cfg is None:
        cfg = resolve_config(get_config())
    set_seed(cfg["seed"])
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("=" * 60)
    print("TRAINING: context image -> TrOCR -> continuation text")
    print("=" * 60)
    print(f"  Device : {dev}")
    print(f"  Model  : {cfg['model_name']}")
    print(f"  Epochs : {cfg['epochs']}  LR={cfg['lr']}")

    ensure_dir(cfg["output_dir"])
    ensure_dir(cfg["debug_dir"])

    # 1. Processor
    print("\n[1/4] Loading processor...")
    proc = build_processor(cfg["model_name"])

    # 2. Data
    print("\n[2/4] Loading data...")
    train_df = pd.read_csv(cfg["train_file"])
    print(f"  {len(train_df)} samples | {train_df['language'].value_counts().to_dict()}")
    tr_df, va_df = stratified_split(train_df, "language", cfg["val_split"], seed=cfg["seed"])
    print(f"  Train: {len(tr_df)} | Val: {len(va_df)}")

    train_ds = OCRDataset(tr_df, proc, cfg, is_test=False)
    val_ds = OCRDataset(va_df, proc, cfg, is_test=False)
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                              num_workers=0, pin_memory=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False,
                            num_workers=0, pin_memory=True, collate_fn=collate_fn)

    # Sanity check
    s = train_ds[0]
    print(f"\n  [SANITY] pixel_values : {tuple(s['pixel_values'].shape)}")
    print(f"  [SANITY] prompt text  : '{s['prompt_text']}'")
    print(f"  [SANITY] target text  : '{s['target_text'][:60]}'")

    # 3. Model
    print("\n[3/4] Building model...")
    mdl = build_model(cfg, proc, dev)
    trainable = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
    total = sum(p.numel() for p in mdl.parameters())
    print(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/max(total,1):.2f}%)")
    total_steps = max(1, (len(train_loader) * cfg["epochs"]) // cfg["grad_accum_steps"])
    optimizer, scheduler = build_optimizer(mdl, cfg, total_steps)

    # 4. Training
    print(f"\n[4/4] Training ({total_steps} optimizer steps)...")
    best_cer = float("inf")
    no_improve = 0
    history = []

    for epoch in range(1, cfg["epochs"] + 1):
        print(f"\n{'=' * 50} Epoch {epoch}/{cfg['epochs']} {'=' * 50}")
        t0 = time.time()
        tr_loss = run_epoch(mdl, train_loader, optimizer, scheduler, cfg, dev, "tr", True)
        va_loss = run_epoch(mdl, val_loader, None, None, cfg, dev, "val", False)
        gen = evaluate_generation(mdl, proc, val_ds, cfg, dev, epoch)
        elapsed = time.time() - t0

        print(f"  train={tr_loss:.4f}  val={va_loss:.4f}  gap={va_loss-tr_loss:+.4f}  {elapsed:.0f}s")
        print(f"  [Gen] CER={gen['cer_mean']:.3f}  pixF1={gen['pixel_f1']:.3f}  "
              f"prefixCER={gen['prefix_cer']:.3f}  repeat={gen['repeat']:.3f}  "
              f"lenRatio={gen['len_ratio']:.2f}  langCons={gen['lang_consistency']:.2f}")
        print(f"  [Dbg] next-token: {gen['debug_first_step']}")

        for lang, cer in gen["cer_per_lang"].items():
            flag = "  !!!" if cer > 0.95 else ""
            pix = gen["pix_per_lang"].get(lang, 0.0)
            print(f"    {lang:>10s}: CER={cer:.3f} pixF1={pix:.3f}{flag}")

        for ex in gen["examples"]:
            print(f"    [{ex['language']}] TRUE: {ex['true'][:60]}")
            print(f"    [{ex['language']}] PRED: {ex['pred'][:60]}")

        history.append({"epoch": epoch, "train": tr_loss, "val": va_loss,
                        "cer": gen["cer_mean"], "pixf1": gen["pixel_f1"]})

        if gen["cer_mean"] < best_cer - 1e-6:
            prev, best_cer = best_cer, gen["cer_mean"]
            save_bundle(mdl, proc, cfg, os.path.join(cfg["output_dir"], "best_model"))
            no_improve = 0
            print(f"  >>> Saved best_model (CER {prev:.4f} → {best_cer:.4f})")
        else:
            no_improve += 1

        if epoch >= cfg["early_stop_warmup_epochs"] and no_improve >= cfg["early_stop_patience"]:
            print(f"  EARLY STOP: {no_improve} epochs without improvement.")
            break

    save_bundle(mdl, proc, cfg, os.path.join(cfg["output_dir"], "final_model"))
    print("\n" + "=" * 60)
    print(f"TRAINING COMPLETE  |  Best CER: {best_cer:.4f}")
    print("=" * 60)
    return history

In [ ]:
# ── Chạy training ─────────────────────────────────────────────
# Bỏ comment dòng dưới để train (yêu cầu train-3.csv)

# history = train(config)

print("Training cell sẵn sàng. Bỏ comment dòng trên để chạy.")

---
## 8. Inference & Submission <a id='8'></a>

Load best checkpoint → generate trên test set → export pixel coordinates → `submission.csv`

In [ ]:
# ============================================================
# inference.py
# ============================================================

def generate_submission(cfg: dict = None):
    if cfg is None:
        cfg = resolve_config(get_config())
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    best_dir = os.path.join(cfg["output_dir"], "best_model")

    if not os.path.exists(best_dir):
        raise FileNotFoundError(f"Không tìm thấy best_model tại {best_dir}. Chạy train() trước.")

    print("=" * 60)
    print("INFERENCE: context image -> TrOCR -> continuation text")
    print("=" * 60)
    print(f"  Device: {dev}")

    # Override config từ checkpoint nếu có
    cfg_path = os.path.join(best_dir, "run_config.json")
    if os.path.exists(cfg_path):
        with open(cfg_path, encoding="utf-8") as f:
            saved_cfg = json.load(f)
        cfg.update(saved_cfg)
        cfg = resolve_config(cfg)

    # 1. Load model
    print(f"\n[1/4] Loading model từ {best_dir}...")
    t0 = time.time()
    mdl, proc = load_bundle(best_dir, dev)
    print(f"  Loaded in {time.time()-t0:.1f}s")

    # 2. Load test data
    print("\n[2/4] Loading test data...")
    test_df = pd.read_csv(cfg["test_file"])
    print(f"  {len(test_df)} samples | {test_df['language'].value_counts().to_dict()}")
    dataset = OCRDataset(test_df, proc, cfg, is_test=True)
    debug_dir = ensure_dir(os.path.join(cfg["debug_dir"], "inference"))

    # 3. Predict
    print(f"\n[3/4] Predicting {len(dataset)} sequences...")
    pixel_list, char_lengths, px_lengths = [], [], []
    t0 = time.time()

    for i in range(len(dataset)):
        sample = dataset[i]
        start_id = int(proc.tokenizer.cls_token_id)
        decoder_prompt_ids = torch.cat([
            torch.tensor([start_id], device=dev, dtype=torch.long),
            sample["prompt_ids"].to(dev),
        ]).unsqueeze(0)

        generated_ids = mdl.generate(
            pixel_values=sample["pixel_values"].unsqueeze(0).to(dev),
            decoder_input_ids=decoder_prompt_ids,
            max_new_tokens=cfg["max_new_tokens"],
            min_new_tokens=cfg["min_new_tokens"],
            num_beams=cfg["num_beams"],
            early_stopping=True,
            repetition_penalty=float(cfg["repetition_penalty"]),
            no_repeat_ngram_size=int(cfg["no_repeat_ngram_size"]),
            length_penalty=float(cfg["length_penalty"]),
        )
        pred_text = clean_prediction(
            proc.batch_decode(generated_ids, skip_special_tokens=True)[0],
            sample["language"],
        )
        pred_img = render_text_to_image(pred_text, sample["max_width"], cfg["font_size"], cfg["padding"])
        pixels = image_to_pixels(pred_img, sample["id"])
        pixel_list.append(pixels)
        char_lengths.append(len(pred_text))
        px_lengths.append(len(pixels))

        if i % 50 == 0:
            elapsed = time.time() - t0
            eta = elapsed / max(i + 1, 1) * (len(dataset) - i - 1)
            print(f"  [{i:>4d}/{len(dataset)}] ETA={eta:.0f}s chars={len(pred_text)} px={len(pixels)}")

        if i < 20:
            pred_img.save(debug_dir / f"pred_{sample['id']}.png")
            render_text_to_image(sample["context_text"], sample["max_width"],
                                  cfg["font_size"], cfg["padding"]).save(debug_dir / f"context_{sample['id']}.png")
            (debug_dir / f"pred_{sample['id']}.txt").write_text(pred_text, encoding="utf-8")

    # 4. Export
    print("\n[4/4] Exporting pixels...")
    all_px = np.concatenate(pixel_list, axis=0)
    arr = np.zeros(len(all_px), dtype=[("sample_id", np.int64), ("row_id", np.int64), ("col_id", np.int64)])
    arr["sample_id"] = all_px[:, 0]
    arr["row_id"] = all_px[:, 1]
    arr["col_id"] = all_px[:, 2]

    ensure_dir("data")
    np.savez("data/pixels.npz", pixels=arr)
    pd.DataFrame({"id": test_df["id"].values, "sample_id": test_df["id"].values}).to_csv("submission.csv", index=False)

    total_px = all_px.shape[0]
    print("\n" + "=" * 60)
    print("INFERENCE COMPLETE")
    print(f"  Predictions  : {len(dataset)}")
    print(f"  Total pixels : {total_px:,}")
    print(f"  Avg px/sample: {total_px / max(len(dataset), 1):.0f}")
    if char_lengths:
        print(f"  Avg chars    : {float(np.mean(char_lengths)):.1f}")
        print(f"  Char range   : {int(np.min(char_lengths))}..{int(np.max(char_lengths))}")
    print("  → data/pixels.npz | submission.csv")
    print("=" * 60)


# ── Chạy inference ────────────────────────────────────────────
# Bỏ comment dòng dưới khi đã có best_model checkpoint

# generate_submission(config)

print("Inference cell sẵn sàng. Bỏ comment dòng trên để chạy.")

---
## Tóm tắt luồng thực thi

```
train-3.csv
    │
    ▼ stratified_split
train_df  val_df
    │
    ▼ OCRDataset
context_text ──► render_text_to_image ──► fit_image_to_canvas
                                              │
                                        pixel_values (3×384×384)
                                              │
                                    VisionEncoderDecoderModel
                                    (TrOCR ViT encoder frozen)
                                    (mBART decoder top-4 layers)
                                              │
                                    compute_weighted_loss
                                    (prompt / prefix / tail)
                                              │
                                    AdamW + WarmupCosine
                                              │
                                    evaluate_generation → CER / PixF1
                                              │
                                    save best_model
                                              │
test.csv ──► generate_submission ──► submission.csv + pixels.npz
```